<a href="https://colab.research.google.com/github/irenearzhang/irenearzhang.github.io/blob/main/modi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
file_path = '/content/Modi statements.xlsx'
df = pd.read_excel(file_path)

In [23]:
for i, url in enumerate(df['URL']):
    url = url.strip()  # Remove leading/trailing whitespace
    url = url.replace('PressReleseDetail', 'PressReleasePage')  # Replace substring
    df.at[i, 'New URL'] = url  # Assign the new URL to the 'New URL' column for the corresponding row

In [25]:
import requests

def check_for_global_south_in_article(article_url, df, index):
    try:
        response = requests.get(article_url)
        response.raise_for_status()  # Will raise an error for bad status codes
        article_html = response.text

        if "Global South" in article_html:
            df.at[index, 'Global_South'] = 1  # Update the specific row and column by name
        else:
            df.at[index, 'Global_South'] = 0
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {article_url}: {e}")
        df.at[index, 'Global_South'] = -1  # Optionally mark it as -1 for error

def check_for_develop_in_article(article_url, df, index):
    try:
        response = requests.get(article_url)
        response.raise_for_status()  # Will raise an error for bad status codes
        article_html = response.text

        if ("developing countr" in article_html) or ("development" in article_html):
            df.at[index, 'Develop'] = 1  # Update the specific row and column by name
        else:
            df.at[index, 'Develop'] = 0
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {article_url}: {e}")
        df.at[index, 'Develop'] = -1  # Optionally mark it as -1 for error

In [26]:
import time

# Loop through the URLs in the DataFrame and update the relevant columns
for index, row in df.iterrows():
    url = row['New URL']
    try:
        check_for_global_south_in_article(url, df, index)
        check_for_develop_in_article(url, df, index)
    except Exception as e:
        print(f"Error processing {url}: {e}")
    time.sleep(1)  # Sleep for 1 second between requests to avoid overwhelming the server

In [ ]:
df.to_excel('/content/my_dataframe.xlsx', index=False, engine='openpyxl')

from google.colab import files
files.download('/content/my_dataframe.xlsx')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
file_path = "/content/Modi statements.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Ensure the date column is in datetime format
df['Date'] = pd.to_datetime(df['Date'])

# Extract the year from the date column
df['Year'] = df['Date'].dt.year

# Group by year and calculate the keyword frequency
keyword_frequency = df.groupby('Year')['Global South'].mean()

# Convert frequency to percentage
keyword_frequency *= 100

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(keyword_frequency.index, keyword_frequency.values, marker='o', label="Keyword Frequency (%)")
plt.title("'Global South' Frequency by Year")
plt.xlabel("Year")
plt.ylabel("Frequency (%)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Group by year and calculate the keyword frequency
keyword_frequency = df.groupby('Year')['Develop'].mean()

# Convert frequency to percentage
keyword_frequency *= 100

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(keyword_frequency.index, keyword_frequency.values, marker='o', label="Keyword Frequency (%)")
plt.title("'Developing Countries' or 'Development' Frequency by Year")
plt.xlabel("Year")
plt.ylabel("Frequency (%)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Group by year and calculate the keyword count
keyword_count = df.groupby('Year')['Global South'].count()

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(keyword_count.index, keyword_count.values, marker='o', label="Statements Count")

for x, y in zip(keyword_count.index, keyword_count.values):
    plt.text(x, y, str(y), fontsize=10, ha='center', va='bottom')

plt.title("Total Number of Statements with 'Global South' by Year")
plt.xlabel("Year")
plt.ylabel("Statements Count")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Group by year and calculate the keyword count
keyword_count = df[df['Develop'] == 1].groupby('Year')['Develop'].count()

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(keyword_count.index, keyword_count.values, marker='o', label="Statements Count")

for x, y in zip(keyword_count.index, keyword_count.values):
    plt.text(x, y, str(y), fontsize=10, ha='center', va='bottom')

plt.title("Total Number of Statements with 'Developing Countries' or 'Development' by Year")
plt.xlabel("Year")
plt.ylabel("Statements Count")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()